In [0]:
%sql
create schema if not exists fabriperezchauca.gold
comment 'OBT y KPIs del negocio, basados en la capa silver';

create or replace view fabriperezchauca.gold.ventas_obt 
comment 'Ventas de silver con columnas de negocio para los KPIs. Grano: una fila = un producto vendido en una fecha' 
as 

with base as (
    select 
        fecha, categoria, producto, talla, cantidad,
        precio_original, precio_venta, descuento, recargo, total,
        forma_pago, observaciones,
        date_trunc('month', fecha) as mes,
        producto <=> 'valor de ventas' as es_resumen_ventas,
        size(collect_set(fecha) over (partition by date_trunc('month', fecha))) as dias_con_ventas_mes
    from fabriperezchauca.silver.ventas
)

select 
    *,
    case when dias_con_ventas_mes >= 20 then 'completo' else 'parcial' end as estado_mes
from base


In [0]:
%sql
select * from fabriperezchauca.gold.ventas_obt limit 100

In [0]:
%sql
select
    mes,
    first(dias_con_ventas_mes) as dias_con_ventas,
    first(estado_mes)          as estado_mes,
    count_if(es_resumen_ventas) as filas_resumen
from fabriperezchauca.gold.ventas_obt
group by mes
order by mes